# SeeSaw — Notebook 2: Gemma 4 1B LoRA Fine-Tuning

Fine-tunes `google/gemma-4-1b-it` on SeeSaw story data using LoRA via Vertex AI.

**Input:** `gs://seesaw-models/training-data/seesaw_beats_train.jsonl`  
**Output:** `gs://seesaw-models/checkpoints/seesaw-gemma4-v1/`

**Runtime:** Vertex AI T4 GPU, ~3 hours, ~$6 USD

See `docs/FINE_TUNING.md` for hyperparameter rationale.

In [ ]:
!pip install transformers peft datasets accelerate trl google-cloud-aiplatform bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

MODEL_NAME   = 'google/gemma-4-1b-it'
OUTPUT_DIR   = '/tmp/seesaw-gemma4-checkpoint'
GCS_BUCKET   = 'gs://seesaw-models/checkpoints/seesaw-gemma4-v1'

# 4-bit quantisation for T4 GPU (16 GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load dataset from GCS
dataset = load_dataset('json', data_files={'train': 'gs://seesaw-models/training-data/seesaw_beats_train.jsonl'})
dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(dataset['train'])}, Eval: {len(dataset['test'])}")

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    fp16=True,
    logging_steps=50,
    save_strategy='epoch',
    evaluation_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    max_seq_length=512,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f'Training complete. Final eval_loss: {trainer.state.best_metric:.4f}')

In [ ]:
# Upload checkpoint to GCS
import subprocess
result = subprocess.run(['gsutil', '-m', 'cp', '-r', OUTPUT_DIR + '/*', GCS_BUCKET], capture_output=True, text=True)
print(result.stdout)
print(f'Checkpoint uploaded to {GCS_BUCKET}')